# 01. Dataset Generation (main_V5)
**Objective:** Compile raw NinaPro DB1 .mat files into a dense, shift-invariant HDF5 tensor, perfectly balanced and ready for Frequency-domain Dictionary Learning.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../'))
from src.config import RAW_DATA_DIR, PREPROCESSED_DIR, MOVEMENT_LABELS
from src.preprocess import load_ninapro_mat, sEMGPreprocessor

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load Raw Data (Subject 1)

In [2]:
# Ensure your raw data is located here: data/raw/Ninapro_DB1/s1/
db_dir = os.path.join(RAW_DATA_DIR, "Ninapro_DB1", "s1")
exercises = ['E1', 'E2', 'E3']

# Offsets to make labels continuous (1-52)
# E1 has 12 movements, E2 has 17 (12+17=29), E3 has 23 (29+23=52)
offsets = {'E1': 0, 'E2': 12, 'E3': 29} 

emg_list, labels_list, reps_list = [], [], []

print("Loading raw .mat files...")
for ex in exercises:
    mat_path = os.path.join(db_dir, f"S1_A1_{ex}.mat")
    if not os.path.exists(mat_path):
        raise FileNotFoundError(f"Missing data file: {mat_path}")
        
    data = load_ninapro_mat(mat_path)
    
    labels = data['labels'].copy()
    active_mask = labels > 0
    labels[active_mask] += offsets[ex]
    
    emg_list.append(data['emg'])
    labels_list.append(labels)
    reps_list.append(data['reps'])

emg_full = np.vstack(emg_list)
labels_full = np.concatenate(labels_list)
reps_full = np.concatenate(reps_list)

print(f"Raw Continuous sEMG Shape: {emg_full.shape}")
print(f"Total Labels: {len(np.unique(labels_full))} (Expected: 53)")

Loading raw .mat files...
Raw Continuous sEMG Shape: (471483, 10)
Total Labels: 53 (Expected: 53)


### 2. Filter, Standardize, and Extract Dense Windows

In [3]:
preprocessor = sEMGPreprocessor(sample_rate=100)

print("\nApplying 1Hz High-Pass Filter...")
filtered_emg = preprocessor.filter_signal(emg_full)

print("Standardizing channels...")
norm_emg = preprocessor.fit_standardize(filtered_emg)

# Extract 200ms windows sliding by 10ms (1 sample)
X_dense, y_dense, reps_dense = preprocessor.extract_dense_windows(norm_emg, labels_full, reps_full, win_size=20)

print("\nBalancing Rest Class...")
X_bal, y_bal, reps_bal = preprocessor.balance_rest_class(X_dense, y_dense, reps_dense)


Applying 1Hz High-Pass Filter...
Standardizing channels...
Extracting dense windows (size=20, stride=1)...

Balancing Rest Class...


### 3. Serialize to HDF5

In [4]:
output_path = os.path.join(PREPROCESSED_DIR, "DB1_S1_Dense.h5")

with h5py.File(output_path, 'w') as f:
    # We use gzip compression because the dense dataset is large
    f.create_dataset('X', data=X_bal, compression='gzip')
    f.create_dataset('y', data=y_bal)
    f.create_dataset('reps', data=reps_bal)

print(f"\nSuccess! Balanced Dense Dataset saved to: {output_path}")
print(f"Final Tensor Shape: {X_bal.shape}")

# Verify the lexicon mapping matches the data
sample_label = y_bal[X_bal.shape[0] // 2]
print(f"Test mapping - Label {sample_label} translates to: '{MOVEMENT_LABELS[sample_label]}'")


Success! Balanced Dense Dataset saved to: /workspaces/TCC/data/preprocessed/DB1_S1_Dense.h5
Final Tensor Shape: (185008, 20, 10)
Test mapping - Label 29 translates to: 'Wrist extension with closed hand'
